<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/add_gemma_eval/model/colabs/gemma3n_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install packages

import builtins

# The Magic Hack: Create a dummy class and inject it into Python's builtins 
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

builtins.PeftConfigLike = DummyPeftConfig

import os
import json
import sys
import torch
from transformers import pipeline
from google.cloud import storage
from huggingface_hub import login

# Import common utils
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.inference_pipeline_runner import run_inference_pipeline

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)



print("Imports finally successful!")

In [ ]:
# --- Configuration ---
import os
# Attempt to fetch the token securely from Google Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    # Fallback to local environment variables or hardcoded string
    HF_TOKEN = os.environ.get("HF_TOKEN", "<YOUR_HUGGING_FACE_TOKEN>")

MODEL_NAME = "google/gemma-3n-e2b-it"
SELECTED_MODEL_KEY = "gemma3n"

GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"
BATCH_SIZE = 4
LIMIT = 10

# Login to Hugging Face
if HF_TOKEN and not HF_TOKEN.startswith("<"):
    login(token=HF_TOKEN)

In [ ]:
#@title Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Gemma model on {device}...")

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_NAME,
    device=device,
    torch_dtype=torch.float32 if device == "cpu" else torch.float16,
)

In [ ]:
#@title Define helper functions for evaluation runner

from common.audio_utils import preprocess_audio_for_model
import os
import torch

def prompt_formatter(entry, local_path):
    """Constructs a simple message structure for Gemma to enforce format."""
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a strict speech-to-text transcriber for emergency radio. Output ONLY the transcribed words. Do not add pleasantries, do not offer help, and do not explain the audio quality."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": local_path},
                {"type": "text", "text": "Transcribe the following audio exactly word-for-word. Do not add any conversational text. If the audio is empty, output nothing."}
            ]
        }
    ]

def find_audio_item(prompt):
    """Helper to find the audio dictionary item in the prompt structure."""
    for message in prompt:
        content = message.get("content", [])
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and item.get("type") == "audio":
                    return item
    return None

def gemma_inference(model, prompts):
    """Runs inference using the pipeline object with greedy decoding."""
    outputs = []
    for p in prompts:
        # Dynamically find the audio item
        audio_item = find_audio_item(p)
        
        if not audio_item:
            logger.warning("No audio item found in prompt!")
            continue
            
        audio_path = audio_item["audio"]
        # audio_path is already the preprocessed WAV file path!
        
        try:
            # Run model (model is the pipe object) with greedy decoding
            out = model(
                text=p, 
                max_new_tokens=100,
                do_sample=False
            )
            outputs.append(out)
            
        except Exception as e:
            logger.error(f"Failed during inference for {audio_path}: {e}")
            outputs.append("[ERROR]")
                
    return outputs

def result_decoder(ans, model):
    """Extracts only the generated transcription from Gemma 3's output."""
    full_text = ans[0]["generated_text"][-1]["content"]
    
    if "<start_of_turn>model\
" in full_text:
        answer = full_text.split("<start_of_turn>model\
")[1]
        answer = answer.replace("<end_of_turn>", "").strip()
        return answer
        
    return full_text.strip()

In [ ]:
#@title Run Evaluation

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_inference_pipeline(
    model=pipe,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=gemma_inference,
    decode_fn=result_decoder,
    preprocess_fn=preprocess_audio_for_model,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

In [ ]:

# Upload results directly to GCS from memory (uncomment to use)
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    SELECTED_MODEL_KEY, 
    EXPERIMENT_NAME, 
    results_list
)